In [41]:
!pip install -q kagglehub beautifulsoup4

import os, re, sys, collections, importlib, warnings
import pandas as pd
from bs4 import BeautifulSoup
import kagglehub
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATASET = "snehaanbhawal/resume-dataset"

def title(t):
    print("\n" + "=" * 8 + " " + t + " " + "=" * 8)

def download():
    """Public datasets usually download without a login. If Kaggle asks, we log in once and retry."""
    try:
        return kagglehub.dataset_download(DATASET)
    except Exception as e:
        print("Direct download needs a Kaggle login ->", str(e)[:150])
        print("Token: kaggle.com > profile picture > Settings > API > Create New Token. Enter username + key below.")
        kagglehub.login()
        return kagglehub.dataset_download(DATASET)

PATH = download()
print("Dataset folder:", PATH)

Using Colab cache for faster access to the 'resume-dataset' dataset.
Dataset folder: /kaggle/input/resume-dataset


In [42]:
title("1. FILES")
all_files = [os.path.join(r, f) for r, _, fs in os.walk(PATH) for f in fs]
print("Total files:", len(all_files), "| by type:", dict(collections.Counter(os.path.splitext(f)[1].lower() for f in all_files)))
print("Top-level items (file counts):", dict(collections.Counter(os.path.relpath(f, PATH).split(os.sep)[0] for f in all_files)))
csvs = sorted(f for f in all_files if f.lower().endswith(".csv"))
for c in csvs:
    print("CSV:", os.path.relpath(c, PATH), "|", round(os.path.getsize(c) / 1e6, 1), "MB")
if not csvs:
    raise RuntimeError("No CSV file found in the dataset folder. Send me the file list printed above.")
pdfs = [f for f in all_files if f.lower().endswith(".pdf")]
pdf_cat = collections.Counter(os.path.basename(os.path.dirname(f)) for f in pdfs)
print("PDF resumes:", len(pdfs), "| PDF folders:", len(pdf_cat))
print("PDFs per folder:", dict(sorted(pdf_cat.items())))
print("Example PDF paths:", [os.path.relpath(p, PATH) for p in pdfs[:3]])


======== 1. FILES ========
Total files: 2485 | by type: {'.csv': 1, '.pdf': 2484}
Top-level items (file counts): {'Resume': 1, 'data': 2484}
CSV: Resume/Resume.csv | 56.3 MB
PDF resumes: 2484 | PDF folders: 24
PDFs per folder: {'ACCOUNTANT': 118, 'ADVOCATE': 118, 'AGRICULTURE': 63, 'APPAREL': 97, 'ARTS': 103, 'AUTOMOBILE': 36, 'AVIATION': 117, 'BANKING': 115, 'BPO': 22, 'BUSINESS-DEVELOPMENT': 120, 'CHEF': 118, 'CONSTRUCTION': 112, 'CONSULTANT': 115, 'DESIGNER': 107, 'DIGITAL-MEDIA': 96, 'ENGINEERING': 118, 'FINANCE': 118, 'FITNESS': 117, 'HEALTHCARE': 115, 'HR': 110, 'INFORMATION-TECHNOLOGY': 120, 'PUBLIC-RELATIONS': 111, 'SALES': 116, 'TEACHER': 102}
Example PDF paths: ['data/data/DESIGNER/22506245.pdf', 'data/data/DESIGNER/13998435.pdf', 'data/data/DESIGNER/67582956.pdf']


In [43]:
CSV = next((c for c in csvs if "resume" in os.path.basename(c).lower()), None) or max(csvs, key=os.path.getsize)

def read(path):
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue

df = read(CSV)

def pick(names):
    low = {c.lower(): c for c in df.columns}
    for n in names:
        if n in low:
            return low[n]

TEXT, HTML, CAT = pick(["resume_str", "resume", "text"]), pick(["resume_html"]), pick(["category", "label"])
title("2. TABLE")
print("File:", os.path.basename(CSV), "| rows, columns:", df.shape)
print("Columns:", list(df.columns))
print("Detected -> text:", TEXT, "| html:", HTML, "| category:", CAT)
if TEXT is None or CAT is None:
    raise RuntimeError("Could not find the text or category column")
print("Missing values:", df.isna().sum().to_dict())
print("Exact duplicate resumes:", int(df[TEXT].duplicated().sum()))


======== 2. TABLE ========
File: Resume.csv | rows, columns: (2484, 4)
Columns: ['ID', 'Resume_str', 'Resume_html', 'Category']
Detected -> text: Resume_str | html: Resume_html | category: Category
Missing values: {'ID': 0, 'Resume_str': 0, 'Resume_html': 0, 'Category': 0}
Exact duplicate resumes: 2


In [44]:
title("3. CATEGORIES")
vc = df[CAT].value_counts()
print("Number of categories:", len(vc))
print(vc.to_string())


======== 3. CATEGORIES ========
Number of categories: 24
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22


In [45]:
title("4. TEXT LENGTH (words per resume)")
txt = df[TEXT].fillna("").astype(str)
words = txt.str.split().str.len()
print(words.describe().round(0).to_string())
print("Under 100 words:", int((words < 100).sum()), "| over 1500 words:", int((words > 1500).sum()))


======== 4. TEXT LENGTH (words per resume) ========
count    2484.0
mean      811.0
std       371.0
min         0.0
25%       651.0
50%       757.0
75%       933.0
max      5190.0
Under 100 words: 1 | over 1500 words: 98


In [46]:
title("5. SAMPLES (first 150 characters of up to 6 resumes)")
for cat, g in list(df.groupby(CAT))[:6]:
    row = g.iloc[0]
    print(f"[{cat}] id={row['ID'] if 'ID' in df.columns else '-'}:", " ".join(str(row[TEXT]).split())[:150])
print("\nOne resume, raw first 700 characters:")
print(repr(txt.iloc[0][:700]))


======== 5. SAMPLES (first 150 characters of up to 6 resumes) ========
[ACCOUNTANT] id=15906625: ACCOUNTANT Professional Summary Current Accountant with the City of Alexandria with over 15 years experience in the accounting industry. Knowledge in 
[ADVOCATE] id=14445309: ADVOCATE Professional Summary Conscientious attorney with experience in government relations, educational law and advocacy for families with special n
[AGRICULTURE] id=54246169: AGRICULTURE ADVISOR AND LANGUAGE OFFICER Professional Summary I successfully completed the 4-years course in Agriculture from Nangarhar Agriculture Fa
[APPAREL] id=24533931: APPAREL ASSOCIATE Summary An organized, detail-oriented, and conscientious self starter, able to learn and adapt quickly to new skills, also able to s
[ARTS] id=31273413: ARTS EDUCATOR Summary Creative and innovative Art teacher passionate about making the middle and high school experience a positive one for all student
[AUTOMOBILE] id=11797122: AUTOMOBILE TRANSPORTER Prof

In [47]:
title("6. SECTION HEADINGS FOUND IN TEXT (% of resumes)")
heads = ["summary", "professional summary", "highlights", "skills", "experience", "work history", "education",
         "accomplishments", "certifications", "languages", "interests", "additional information", "objective", "projects"]
low = txt.str.lower()
for h in heads:
    print(f"{h:26s}", round(100 * low.str.contains(r"\b" + re.escape(h) + r"\b").mean(), 1), "%")


======== 6. SECTION HEADINGS FOUND IN TEXT (% of resumes) ========
summary                    76.7 %
professional summary       18.9 %
highlights                 43.8 %
skills                     99.0 %
experience                 96.9 %
work history               12.5 %
education                  98.1 %
accomplishments            38.8 %
certifications             15.9 %
languages                  12.5 %
interests                  12.0 %
additional information     19.1 %
objective                  4.3 %
projects                   42.4 %


In [48]:
title("7. HTML STRUCTURE (first 200 resumes)")
if HTML:
    tags, classes = collections.Counter(), collections.Counter()
    for h in df[HTML].dropna().astype(str).head(200):
        for el in BeautifulSoup(h, "html.parser").find_all(True):
            tags[el.name] += 1
            for c in (el.get("class") or []):
                classes[c] += 1
    print("Top tags:", tags.most_common(12))
    print("Top CSS classes:", classes.most_common(30))
    print("Section-like classes:", [c for c in classes if any(k in c.lower() for k in ("section", "title", "head"))][:20])
    print("HTML sample (first 600 characters):", str(df[HTML].dropna().iloc[0])[:600])
else:
    print("No HTML column found")


======== 7. HTML STRUCTURE (first 200 resumes) ========
Top tags: [('span', 19641), ('div', 9425), ('li', 7871), ('ul', 1288), ('br', 901), ('p', 757), ('td', 338), ('table', 169), ('tbody', 169), ('tr', 169), ('font', 163), ('strong', 90)]
Top CSS classes: [('paddedline', 3834), ('joblocation', 3012), ('jobdates', 2380), ('paragraph', 2374), ('singlecolumn', 2173), ('field', 1821), ('jobstate', 1533), ('section', 1500), ('firstparagraph', 1403), ('companyname', 1294), ('jobcity', 1262), ('heading', 1202), ('sectiontitle', 1202), ('jobline', 951), ('jobtitle', 932), ('datesWrapper', 784), ('degree', 416), ('programline', 416), ('dates_wrapper', 387), ('companyname_educ', 381), ('thinbottomborder', 346), ('fontsize', 200), ('fontface', 200), ('pagesize', 200), ('firstsection', 200), ('vmargins', 199), ('hmargins', 199), ('PARAGRAPH_NAME', 199), ('name', 199), ('linespacing', 194)]
Section-like classes: ['section', 'firstsection', 'heading', 'sectiontitle', 'jobtitle', 'SECTION_NAME', '

In [49]:
title("8. IDENTITY CLUES IN TEXT (% of resumes containing them)")
checks = {"email": r"[\w.+-]+@[\w-]+\.[\w.]+", "phone": r"\+?\d[\d\s().-]{8,}\d", "year 1950-2030": r"\b(19[5-9]\d|20[0-3]\d)\b",
          "he/she/his/her": r"\b(he|she|his|her|him)\b", "university/college": r"\b(university|college|institute)\b",
          "city, ST": r"\b[A-Z][a-z]+,\s?[A-Z]{2}\b"}
for name, pat in checks.items():
    rx = re.compile(pat, 0 if name == "city, ST" else re.I)
    print(f"{name:22s}", round(100 * txt.str.contains(rx).mean(), 1), "%")


======== 8. IDENTITY CLUES IN TEXT (% of resumes containing them) ========
email                  0.8 %
phone                  14.5 %
year 1950-2030         99.7 %
he/she/his/her         5.5 %
university/college     89.7 %
city, ST               33.1 %


In [50]:
title("9. ENVIRONMENT")
print("Python:", sys.version.split()[0])
try:
    import torch
    print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)
for m in ("spacy", "sklearn", "shap", "transformers", "sentence_transformers", "pdfplumber"):
    try:
        print(m, getattr(importlib.import_module(m), "__version__", "installed"))
    except Exception:
        print(m, "-> not installed yet")

print("\nDONE")


======== 9. ENVIRONMENT ========
Python: 3.13.16
torch: 2.11.0+cpu | GPU: False 
spacy 3.8.16
sklearn 1.6.1
shap 0.54.1
transformers 5.20.1
sentence_transformers 5.7.0
pdfplumber 0.11.10

DONE


In [51]:
!pip install -q kagglehub beautifulsoup4 pdfplumber

import os, re, collections, warnings
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import kagglehub
warnings.filterwarnings("ignore")

SEED = 42
OUT = "/content/fairforge_data"
os.makedirs(OUT, exist_ok=True)

In [52]:
!pip install -q kagglehub beautifulsoup4

In [53]:
PATH = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
CSV = next(os.path.join(r, f) for r, _, fs in os.walk(PATH) for f in fs if f.lower().endswith(".csv"))
try:
    df = pd.read_csv(CSV)
except UnicodeDecodeError:
    df = pd.read_csv(CSV, encoding="latin-1")

def squash(s):
    """Collapse all whitespace (spaces, tabs, new lines) into single spaces."""
    return re.sub(r"\s+", " ", str(s)).strip()

Using Colab cache for faster access to the 'resume-dataset' dataset.


In [54]:
df["text_clean"] = df["Resume_str"].fillna("").map(squash)
n0 = len(df)
df = df[df["text_clean"].str.split().str.len() >= 50]          # drop empty / tiny resumes
df = df.drop_duplicates("text_clean").reset_index(drop=True)    # drop exact duplicates
print(f"Rows: {n0} -> {len(df)} after removing empty and duplicate resumes")

Rows: 2484 -> 2481 after removing empty and duplicate resumes


In [55]:
RULES = [("SUMMARY", ("summary", "profile", "objective", "about")),
         ("SKILLS", ("skill", "highlight", "qualification", "competenc", "expertise", "strength")),
         ("EXPERIENCE", ("experience", "employment", "work history", "career", "professional history")),
         ("EDUCATION", ("education", "training", "academic", "degree"))]

def canon(title):
    """Map a raw section title (e.g. 'Work History') to one of our 6 labels."""
    t = title.lower()
    for label, keys in RULES:
        if any(k in t for k in keys):
            return label
    return "OTHER"

BLOCK = re.compile(r"(?i)<br\s*/?>|</(?:div|p|li|tr|ul|table|h\d)>")

def html_lines(html):
    """Return (section number, raw title, label, line) for every text line of one resume."""
    soup = BeautifulSoup(str(html), "html.parser")
    secs = [s for s in soup.select("div.section") if not s.find_parent("div", class_="section")]
    out = []
    for k, sec in enumerate(secs):
        t = sec.select_one(".sectiontitle")
        raw = squash(t.get_text()) if t else ""
        label = canon(raw) if raw else "HEADER"                 # the untitled first block = job title / contact
        html_sec = re.sub(r"(?i)</(?:span|td)>", " ", str(sec))            # keep neighbouring spans from gluing words together
        text = BeautifulSoup(BLOCK.sub("\n", html_sec), "html.parser").get_text()
        lines = [squash(x) for x in text.split("\n")]
        lines = [x for x in lines if len(x) > 1]
        if raw and lines and lines[0].lower() == raw.lower():   # remove the heading itself
            lines = lines[1:]
        out += [(k, raw or "(none)", label, x) for x in lines]
    return out

rows, no_sections = [], 0
for rid, cat, html in tqdm(zip(df["ID"], df["Category"], df["Resume_html"]), total=len(df), desc="Reading HTML"):
    found = html_lines(html)
    if not found:
        no_sections += 1
    rows += [(rid, cat) + r for r in found]
lines_df = pd.DataFrame(rows, columns=["ID", "Category", "sec_no", "sec_raw", "section", "line"])

Reading HTML:   0%|          | 0/2481 [00:00<?, ?it/s]

In [56]:
ids = df[["ID", "Category"]]
train, temp = train_test_split(ids, test_size=0.30, stratify=ids["Category"], random_state=SEED)
val, test = train_test_split(temp, test_size=0.50, stratify=temp["Category"], random_state=SEED)
split_of = {**{i: "train" for i in train["ID"]}, **{i: "val" for i in val["ID"]}, **{i: "test" for i in test["ID"]}}
df["split"] = df["ID"].map(split_of)
lines_df["split"] = lines_df["ID"].map(split_of)

In [57]:
df[["ID", "Category", "split", "text_clean"]].to_csv(f"{OUT}/resumes_clean.csv", index=False)
lines_df.to_csv(f"{OUT}/resume_lines.csv.gz", index=False, compression="gzip")
print("Saved to", OUT, "->", {f: f"{os.path.getsize(os.path.join(OUT, f)) / 1e6:.1f} MB" for f in os.listdir(OUT)})

Saved to /content/fairforge_data -> {'emb_anon.npy': '3.8 MB', 'career_vecs.npy': '1.3 MB', 'emb_raw.npy': '3.8 MB', 'resume_order.csv': '0.1 MB', 'resumes_anon.csv': '28.1 MB', 'resumes_clean.csv': '14.8 MB', 'lines_anon.csv.gz': '6.2 MB', 'resume_lines.csv.gz': '5.1 MB', 'section_stats.npy': '0.1 MB'}


In [58]:
def title(t):
    print("\n" + "=" * 8 + " " + t + " " + "=" * 8)

title("A. SPLIT SIZES")
print(df["split"].value_counts().to_dict(), "| resumes with no HTML sections found:", no_sections)

title("B. LINES PER LABEL")
print(lines_df["section"].value_counts().to_string())
print("Average lines per resume:", round(len(lines_df) / df["ID"].nunique(), 1))

title("C. TOP 30 RAW SECTION TITLES -> OUR LABEL")
top = lines_df.groupby(["sec_raw", "section"]).ID.nunique().sort_values(ascending=False).head(30)
for (raw, lab), n in top.items():
    print(f"{n:5d} resumes | {raw[:40]:40s} -> {lab}")

title("D. % OF RESUMES THAT HAVE EACH LABEL")
have = lines_df.groupby("section").ID.nunique() / df["ID"].nunique() * 100
print(have.round(1).to_string())

title("E. EXAMPLE LINES")
for lab in ["HEADER", "SUMMARY", "SKILLS", "EXPERIENCE", "EDUCATION", "OTHER"]:
    sample = lines_df[lines_df["section"] == lab]["line"].head(3).tolist()
    print(f"[{lab}]", [s[:90] for s in sample])

title("F. YEARS INSIDE LINES")
for lab in ["EDUCATION", "EXPERIENCE"]:
    sub = lines_df[lines_df["section"] == lab]["line"]
    print(lab, "lines containing a year:", round(100 * sub.str.contains(r"\b(?:19|20)\d{2}\b").mean(), 1), "%")

print("\nDONE")


======== A. SPLIT SIZES ========
{'train': 1736, 'test': 373, 'val': 372} | resumes with no HTML sections found: 0

======== B. LINES PER LABEL ========
section
EXPERIENCE    90298
SKILLS        25242
OTHER         11235
EDUCATION      8921
SUMMARY        4353
HEADER         2543
Average lines per resume: 57.5

======== C. TOP 30 RAW SECTION TITLES -> OUR LABEL ========
 2473 resumes | (none)                                   -> HEADER
 2308 resumes | Skills                                   -> SKILLS
 1904 resumes | Education                                -> EDUCATION
 1705 resumes | Experience                               -> EXPERIENCE
 1343 resumes | Summary                                  -> SUMMARY
  864 resumes | Highlights                               -> SKILLS
  756 resumes | Accomplishments                          -> OTHER
  508 resumes | Education and Training                   -> EDUCATION
  451 resumes | Additional Information                   -> OTHER
  444 resumes 

In [59]:
!pip install -q spacy scikit-learn joblib tqdm
!python -m spacy download en_core_web_sm -q

import os, re, time, collections, warnings
import pandas as pd
import spacy, joblib
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings("ignore")

DATA, MODELS = "/content/fairforge_data", "/content/fairforge_models"
os.makedirs(MODELS, exist_ok=True)
if not os.path.exists(f"{DATA}/resume_lines.csv.gz"):
    raise RuntimeError("Step 2 files are missing")

def squash(s):
    return re.sub(r"\s+", " ", str(s)).strip()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [60]:
res = pd.read_csv(f"{DATA}/resumes_clean.csv")
lines = pd.read_csv(f"{DATA}/resume_lines.csv.gz")
lines["line"] = lines["line"].astype(str)
low = lines["sec_raw"].astype(str).str.lower()
lines.loc[low.str.contains("personal information"), "section"] = "HEADER"          # contact details, not skills
lines.loc[low.str.contains("career overview|career focus"), "section"] = "SUMMARY"

In [61]:
ARTIFACT = re.compile(r"Company Name|City\s*,\s*State|[－—–]")
lines["line"] = lines["line"].map(lambda s: squash(ARTIFACT.sub(" ", s)))
lines = lines[lines["line"].str.len() > 1].reset_index(drop=True)
print("Lines to process:", len(lines), "| resumes:", lines["ID"].nunique())

Lines to process: 141472 | resumes: 2481


In [62]:
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
ENT_TAG = {"PERSON": "[NAME]", "GPE": "[PLACE]", "LOC": "[PLACE]", "FAC": "[PLACE]"}
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
URL = re.compile(r"(?:https?://|www\.)\S+", re.I)
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")
YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
PRONOUN = re.compile(r"\b(?:he|she|him|her|his|hers|himself|herself)\b", re.I)
TITLE = re.compile(r"\b(?:Mr|Mrs|Ms|Miss)\.?(?=\s)")
COLLEGE = re.compile(r"\b(?:[A-Z][\w&'.-]*\s+){0,4}(?:University|College|Institute)(?:\s+of\s+(?:[A-Z][\w&'.-]*\s*){1,4})?")

counts, seen = collections.Counter(), {"[NAME]": collections.Counter(), "[PLACE]": collections.Counter()}

def anonymize(texts, sections):
    out = []
    for doc, sec in zip(nlp.pipe(texts, batch_size=1000), sections):
        t = doc.text
        spans = sorted(((e.start_char, e.end_char, ENT_TAG[e.label_]) for e in doc.ents if e.label_ in ENT_TAG), reverse=True)
        for s, e, tag in spans:
            seen[tag][t[s:e]] += 1
            counts[tag] += 1
            t = t[:s] + tag + t[e:]
        rules = [(EMAIL, "[EMAIL]"), (URL, "[URL]"), (PHONE, "[PHONE]"), (COLLEGE, "[COLLEGE]"), (PRONOUN, "[PRONOUN]"), (TITLE, "[TITLE]")]
        if sec == "EDUCATION":                       # hide graduation years, but keep job dates so experience can still be counted
            rules.append((YEAR, "[YEAR]"))
        for rx, tag in rules:
            t, n = rx.subn(tag, t)
            counts[tag] += n
        out.append(t)
    return out

t0 = time.time()
texts, secs, done = lines["line"].tolist(), lines["section"].tolist(), []
CHUNK = 20000
for i in tqdm(range(0, len(texts), CHUNK), desc="Hiding identity"):
    done += anonymize(texts[i:i + CHUNK], secs[i:i + CHUNK])
lines["line_anon"] = done
print(f"Anonymized {len(lines)} lines in {time.time() - t0:.0f} seconds")

Hiding identity:   0%|          | 0/8 [00:00<?, ?it/s]

Anonymized 141472 lines in 148 seconds


In [63]:
use = lines[lines["section"] != "HEADER"]
res = res.set_index("ID")
res["text_raw"] = use.groupby("ID")["line"].apply(" ".join)
res["text_anon"] = use.groupby("ID")["line_anon"].apply(" ".join)
res = res.dropna(subset=["text_raw", "text_anon"]).reset_index()

lines.to_csv(f"{DATA}/lines_anon.csv.gz", index=False, compression="gzip")
res[["ID", "Category", "split", "text_raw", "text_anon"]].to_csv(f"{DATA}/resumes_anon.csv", index=False)

In [64]:
def run(col):
    tr, te = res[res["split"] == "train"], res[res["split"] == "test"]
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, stop_words="english")
    Xtr, Xte = vec.fit_transform(tr[col]), vec.transform(te[col])
    clf = LogisticRegression(max_iter=300, C=10, class_weight="balanced").fit(Xtr, tr["Category"])
    pred = clf.predict(Xte)
    return vec, clf, te, pred

results = {}
for col in ("text_raw", "text_anon"):
    t0 = time.time()
    vec, clf, te, pred = run(col)
    results[col] = (accuracy_score(te["Category"], pred), f1_score(te["Category"], pred, average="macro"), te, pred)
    joblib.dump({"vectorizer": vec, "model": clf}, f"{MODELS}/baseline_{col.split('_')[1]}.joblib")
    print(f"Trained baseline on {col} in {time.time() - t0:.0f}s")

Trained baseline on text_raw in 17s
Trained baseline on text_anon in 17s


In [65]:
def title(t):
    print("\n" + "=" * 8 + " " + t + " " + "=" * 8)

title("A. WHAT THE ANONYMIZER HID")
print(dict(counts))
for tag in ("[NAME]", "[PLACE]"):
    print(f"Most common text hidden as {tag}:", seen[tag].most_common(15))

title("B. BEFORE / AFTER EXAMPLES (8 random changed lines)")
changed = lines[lines["line"] != lines["line_anon"]].sample(min(8, int((lines["line"] != lines["line_anon"]).sum())), random_state=1)
for _, r in changed.iterrows():
    print(f"[{r['section']}] BEFORE: {r['line'][:110]}\n{'':13s}AFTER : {r['line_anon'][:110]}")

title("C. BASELINE ML MODEL: guess the job category from the resume (test set)")
for col, (acc, f1, _, _) in results.items():
    print(f"{col:10s} accuracy = {acc:.3f} | macro F1 = {f1:.3f}")
print("Accuracy change after hiding identity:", f"{results['text_anon'][0] - results['text_raw'][0]:+.3f}")

title("D. WEAKEST 5 CATEGORIES (anonymized text, F1 score)")
acc, f1, te, pred = results["text_anon"]
rep = classification_report(te["Category"], pred, output_dict=True, zero_division=0)
weak = sorted(((k, v["f1-score"], int(v["support"])) for k, v in rep.items() if k in set(te["Category"])), key=lambda x: x[1])[:5]
for k, f, n in weak:
    print(f"{k:24s} F1 = {f:.2f} (test resumes: {n})")

title("E. SAVED FILES")
for d in (DATA, MODELS):
    print(d, {f: f"{os.path.getsize(os.path.join(d, f)) / 1e6:.1f} MB" for f in sorted(os.listdir(d))})

print("\nDONE")


======== A. WHAT THE ANONYMIZER HID ========
{'[EMAIL]': 35, '[URL]': 194, '[PHONE]': 531, '[COLLEGE]': 4972, '[PRONOUN]': 345, '[TITLE]': 117, '[PLACE]': 12096, '[NAME]': 15218, '[YEAR]': 4767}
Most common text hidden as [NAME]: [('Detail', 211), ('Visio', 121), ('Oracle', 121), ('Dean', 120), ('Maintained', 114), ('Developed', 113), ('Java', 102), ('Liaison', 96), ('Oversaw', 93), ('Active Directory', 92), ('Volunteer', 83), ('Twitter', 73), ('Supervisor', 71), ('C++', 71), ('Mac', 63)]
Most common text hidden as [PLACE]: [('US', 468), ('United States', 162), ('U.S.', 142), ('India', 137), ('Cisco', 137), ('USA', 122), ('Texas', 106), ('Windows', 88), ('Client', 78), ('California', 77), ('Florida', 70), ('Linux', 68), ('the United States', 68), ('New York', 67), ('Chicago', 65)]

======== B. BEFORE / AFTER EXAMPLES (8 random changed lines) ========
[OTHER] BEFORE: Asphalt Paving Technician, Level 1: Florida Department of Transportation (May 2005) -Earthwork Construction In
         

In [66]:
import os, re, time, collections, warnings
import pandas as pd
import spacy
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

DATA = "/content/fairforge_data"
lines = pd.read_csv(f"{DATA}/lines_anon.csv.gz")
lines["line"], lines["line_anon"] = lines["line"].astype(str), lines["line_anon"].astype(str)

nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
URL = re.compile(r"(?:https?://|www\.)\S+", re.I)
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")
YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
PRONOUN = re.compile(r"\b(?:he|she|him|her|his|hers|himself|herself)\b", re.I)
TITLE = re.compile(r"\b(?:Mr|Mrs|Ms|Miss)\.?(?=\s)")
COLLEGE = re.compile(r"\b(?:[A-Z][\w&'.-]*\s+){0,4}(?:University|College|Institute)(?:\s+of\s+(?:[A-Z][\w&'.-]*\s*){1,4})?")

# A person name must look like "First Last" (or "First M. Last") and must not contain job / technology words.
NAME_SHAPE = re.compile(r"[A-Z][a-z]+(?:\s+[A-Z]\.?)?(?:\s+[A-Z][a-z]+)+")
NOT_NAME = set("""active directory microsoft office visual studio customer service services project management business data systems
system network web design sales account accounting financial health care medical human resources quality control assurance software
hardware technical support help desk team lead senior junior general operations administrative executive public relations social media
oracle java python cisco windows linux client detail liaison supervisor volunteer maintained developed oversaw""".split())
NOT_PLACE = {"cisco", "windows", "linux", "client", "mac", "oracle", "java", "visio", "twitter", "facebook", "linkedin"}

counts, seen = collections.Counter(), {"[NAME]": collections.Counter(), "[PLACE]": collections.Counter()}

def keep_entity(ent):
    txt = ent.text.strip()
    if ent.label_ == "PERSON":
        return bool(NAME_SHAPE.fullmatch(txt)) and not any(w.lower() in NOT_NAME for w in txt.split()), "[NAME]"
    if ent.label_ == "GPE":                               # LOC and FAC are dropped: they were mostly products (Windows, Linux)
        return txt.lower() not in NOT_PLACE, "[PLACE]"
    return False, ""

def anonymize(texts, sections):
    out = []
    for doc, sec in zip(nlp.pipe(texts, batch_size=1000), sections):
        t = doc.text
        spans = []
        for e in doc.ents:
            ok, tag = keep_entity(e)
            if ok:
                spans.append((e.start_char, e.end_char, tag))
        for s, e, tag in sorted(spans, reverse=True):
            seen[tag][t[s:e]] += 1
            counts[tag] += 1
            t = t[:s] + tag + t[e:]
        rules = [(EMAIL, "[EMAIL]"), (URL, "[URL]"), (PHONE, "[PHONE]"), (COLLEGE, "[COLLEGE]"), (PRONOUN, "[PRONOUN]"), (TITLE, "[TITLE]")]
        if sec == "EDUCATION":
            rules.append((YEAR, "[YEAR]"))
        for rx, tag in rules:
            t, n = rx.subn(tag, t)
            counts[tag] += n
        out.append(t)
    return out


todo = lines["line_anon"].str.contains(r"\[(?:NAME|PLACE)\]", regex=True)
idx = lines.index[todo]
print("Lines to redo:", len(idx), "of", len(lines))
t0 = time.time()
raw, secs, new = lines.loc[idx, "line"].tolist(), lines.loc[idx, "section"].tolist(), []
for i in tqdm(range(0, len(raw), 20000), desc="Redoing"):
    new += anonymize(raw[i:i + 20000], secs[i:i + 20000])
lines.loc[idx, "line_anon"] = new
print(f"Done in {time.time() - t0:.0f}s")

# rebuild the resume-level text
res = pd.read_csv(f"{DATA}/resumes_anon.csv").set_index("ID")
use = lines[lines["section"] != "HEADER"]
res["text_anon"] = use.groupby("ID")["line_anon"].apply(" ".join)
res = res.dropna(subset=["text_raw", "text_anon"]).reset_index()
lines.to_csv(f"{DATA}/lines_anon.csv.gz", index=False, compression="gzip")
res.to_csv(f"{DATA}/resumes_anon.csv", index=False)

print("\n======== WHAT THE FIXED ANONYMIZER HID (this pass only) ========")
print(dict(counts))
for tag in ("[NAME]", "[PLACE]"):
    print(f"Most common text hidden as {tag}:", seen[tag].most_common(15))
print("\nStill-masked lines left in total: names/places =", int(lines["line_anon"].str.contains(r"\[(?:NAME|PLACE)\]", regex=True).sum()))
print("\nDONE")

Lines to redo: 16982 of 141472


Redoing:   0%|          | 0/1 [00:00<?, ?it/s]

Done in 38s

======== WHAT THE FIXED ANONYMIZER HID (this pass only) ========
{'[PLACE]': 10087, '[EMAIL]': 26, '[URL]': 47, '[PHONE]': 310, '[COLLEGE]': 1421, '[PRONOUN]': 148, '[TITLE]': 110, '[NAME]': 4783, '[YEAR]': 1469}
Most common text hidden as [NAME]: [('Problem Solving', 54), ('Adobe Acrobat', 44), ('Magna Cum Laude', 34), ('Creative Suite', 34), ('Balance Sheet', 33), ('Sarbanes Oxley', 25), ('Adobe Photoshop', 23), ('Summa Cum Laude', 20), ('Creative Writing', 20), ('Sous Chef', 20), ('Leadership Skills', 19), ('Graphic Designer', 19), ('Quick Books', 19), ('People Soft', 16), ('Problem Resolution', 15)]
Most common text hidden as [PLACE]: [('US', 468), ('United States', 162), ('U.S.', 142), ('India', 137), ('USA', 122), ('Texas', 106), ('California', 77), ('Florida', 70), ('the United States', 68), ('New York', 67), ('Chicago', 65), ('Dallas', 65), ('China', 62), ('Canada', 59), ('Oversaw', 54)]

Still-masked lines left in total: names/places = 10179

DONE


In [67]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
import os, re, sys, json, time, collections, warnings, subprocess
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
warnings.filterwarnings("ignore")
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [68]:
import spacy
from tqdm.auto import tqdm
DATA = "/content/fairforge_data"
_l = pd.read_csv(f"{DATA}/lines_anon.csv.gz")
_l["line"], _l["line_anon"] = _l["line"].astype(str), _l["line_anon"].astype(str)
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
URL = re.compile(r"(?:https?://|www\.)\S+", re.I)
PHONE = re.compile(r"\+?\d[\d\s().-]{8,}\d")
YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
PRONOUN = re.compile(r"\b(?:he|she|him|her|his|hers|himself|herself)\b", re.I)
TITLE = re.compile(r"\b(?:Mr|Mrs|Ms|Miss)\.?(?=\s)")
COLLEGE = re.compile(r"\b(?:[A-Z][\w&'.-]*\s+){0,4}(?:University|College|Institute)(?:\s+of\s+(?:[A-Z][\w&'.-]*\s*){1,4})?")
FIRST_NAMES = set("""emily sarah jessica ashley amanda jennifer lisa mary linda patricia susan karen nancy laura emma olivia sophia hannah megan
rachel lauren anna maria michelle kimberly donna carol sandra ruth sharon deborah cynthia angela melissa brenda amy katherine christine
samantha nicole elizabeth stephanie rebecca heather james john robert michael william david richard joseph thomas charles christopher
daniel matthew anthony mark donald steven paul andrew joshua kenneth kevin brian george edward ronald timothy jason jeffrey ryan jacob
gary nicholas eric jonathan stephen larry justin scott brandon benjamin samuel gregory alexander patrick frank raymond jack dennis jerry
tyler aaron jose adam nathan henry peter rahul priya amit anil sunil suresh ramesh rajesh vijay sanjay ajay manoj deepak arun ravi kiran
anita sunita sita geeta meena lakshmi divya pooja neha kavita anjali swati rohit vikram arjun karthik krishna ganesh mohan naveen prakash
srinivas venkat aditya naman jeyaditya prasath ayesha fatima mohammed muhammad ahmed ali omar imran zainab nikhil varun siddharth harish
mahesh pradeep sandeep ashok rakesh vinod anand shruti nisha sneha preeti rekha usha radha shweta""".split())
NOT_NAME = set("""active directory microsoft office visual studio customer service services project management business data systems system
network web design sales account accounting financial health care medical human resources quality control assurance software hardware
technical support help desk team lead senior junior general operations administrative executive public relations social media oracle java
python cisco windows linux client""".split())
NOT_PLACE = set("""cisco windows linux client mac oracle java visio twitter facebook linkedin oversaw managed maintained developed supervised led
trained assisted coordinated responsible performed handled provided created implemented""".split())
counts, seen = collections.Counter(), {"[NAME]": collections.Counter(), "[PLACE]": collections.Counter()}

def keep_entity(ent):
    txt = ent.text.strip()
    words = txt.replace(".", " ").split()
    if ent.label_ == "PERSON":
        ok = bool(words) and words[0].lower() in FIRST_NAMES and not any(w.lower() in NOT_NAME for w in words[1:])
        return ok, "[NAME]"
    if ent.label_ == "GPE":
        return txt.lower() not in NOT_PLACE, "[PLACE]"
    return False, ""

def anonymize(texts, sections):
    out = []
    for doc, sec in zip(nlp.pipe(texts, batch_size=1000), sections):
        t = doc.text
        spans = []
        for e in doc.ents:
            ok, tag = keep_entity(e)
            if ok:
                spans.append((e.start_char, e.end_char, tag))
        for s, e, tag in sorted(spans, reverse=True):
            seen[tag][t[s:e]] += 1
            counts[tag] += 1
            t = t[:s] + tag + t[e:]
        rules = [(EMAIL, "[EMAIL]"), (URL, "[URL]"), (PHONE, "[PHONE]"), (COLLEGE, "[COLLEGE]"), (PRONOUN, "[PRONOUN]"), (TITLE, "[TITLE]")]
        if sec == "EDUCATION":
            rules.append((YEAR, "[YEAR]"))
        for rx, tag in rules:
            t, n = rx.subn(tag, t)
            counts[tag] += n
        out.append(t)
    return out

_idx = _l.index[_l["line_anon"].str.contains(r"\[(?:NAME|PLACE)\]", regex=True)]
print("PART 0: redoing", len(_idx), "lines that were masked as a name or place")
_raw, _sec, _new = _l.loc[_idx, "line"].tolist(), _l.loc[_idx, "section"].tolist(), []
for i in tqdm(range(0, len(_raw), 20000), desc="Redoing"):
    _new += anonymize(_raw[i:i + 20000], _sec[i:i + 20000])
_l.loc[_idx, "line_anon"] = _new
_r = pd.read_csv(f"{DATA}/resumes_anon.csv").set_index("ID")
_r["text_anon"] = _l[_l["section"] != "HEADER"].groupby("ID")["line_anon"].apply(" ".join)
_r = _r.dropna(subset=["text_raw", "text_anon"]).reset_index()
_l.to_csv(f"{DATA}/lines_anon.csv.gz", index=False, compression="gzip")
_r.to_csv(f"{DATA}/resumes_anon.csv", index=False)
print("Masks added in this pass:", {k: v for k, v in counts.items() if v})
for tag in ("[NAME]", "[PLACE]"):
    print(f"Text hidden as {tag}:", seen[tag].most_common(12))

PART 0: redoing 10179 lines that were masked as a name or place


Redoing:   0%|          | 0/1 [00:00<?, ?it/s]

Masks added in this pass: {'[PLACE]': 9981, '[EMAIL]': 22, '[URL]': 27, '[PHONE]': 239, '[COLLEGE]': 1199, '[PRONOUN]': 92, '[TITLE]': 116, '[YEAR]': 1139, '[NAME]': 348}
Text hidden as [NAME]: [('Kimberly Malone', 4), ('Thomas Edison Elementary', 4), ('Charles', 4), ('Lisa', 3), ('John Deere', 3), ('Thomas Jefferson High School', 3), ('Larry Hertzler', 3), ('Cynthia', 3), ('Anthony Nelson', 2), ('David Majcher', 2), ('Jerry Russell &', 2), ('William Woods', 2)]
Text hidden as [PLACE]: [('US', 468), ('United States', 162), ('U.S.', 142), ('India', 137), ('USA', 122), ('Texas', 106), ('California', 77), ('Florida', 70), ('the United States', 68), ('New York', 67), ('Chicago', 65), ('Dallas', 65)]


In [69]:
DATA, MODELS = "/content/fairforge_data", "/content/fairforge_models"
for f in ("lines_anon.csv.gz", "resumes_anon.csv"):
    if not os.path.exists(f"{DATA}/{f}"):
        raise RuntimeError(f"{f} is missing. Run Step 3 first, then run this cell.")
os.makedirs(MODELS, exist_ok=True)
torch.manual_seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

res = pd.read_csv(f"{DATA}/resumes_anon.csv")
lines = pd.read_csv(f"{DATA}/lines_anon.csv.gz")
lines["line_anon"] = lines["line_anon"].astype(str)
res["text_raw"], res["text_anon"] = res["text_raw"].fillna("").astype(str), res["text_anon"].fillna("").astype(str)
LABELS = ["HEADER", "SUMMARY", "SKILLS", "EXPERIENCE", "EDUCATION", "OTHER"]
CATS = sorted(res["Category"].unique())
pos = {rid: i for i, rid in enumerate(res["ID"])}          # row number of every resume (keeps all saved arrays aligned)

Device: cpu


In [30]:
t0 = time.time()
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
emb_raw = st.encode(res["text_raw"].tolist(), batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
emb_anon = st.encode(res["text_anon"].tolist(), batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
np.save(f"{DATA}/emb_raw.npy", emb_raw); np.save(f"{DATA}/emb_anon.npy", emb_anon)
res[["ID", "Category", "split"]].to_csv(f"{DATA}/resume_order.csv", index=False)
print(f"MiniLM embeddings done in {time.time() - t0:.0f}s | shape {emb_anon.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

MiniLM embeddings done in 465s | shape (2481, 384)


In [31]:
TOK = re.compile(r"\[[a-z]+\]|[a-z0-9+#.]+")
def toks(s):
    return TOK.findall(str(s).lower())

train_lines = lines[lines["split"] == "train"]
cnt = collections.Counter(w for s in train_lines["line_anon"] for w in toks(s))
vocab = {"<pad>": 0, "<unk>": 1}
for w, c in cnt.most_common(30000):
    if c >= 3:
        vocab[w] = len(vocab)
json.dump(vocab, open(f"{MODELS}/vocab.json", "w"))
print("Vocabulary size:", len(vocab))

def encode(texts, maxlen):
    arr = np.zeros((len(texts), maxlen), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, 1) for w in toks(t)][:maxlen]
        arr[i, :len(ids)] = ids
    return arr

def predict(model, X, bs=2048):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(model(torch.tensor(X[i:i + bs]).to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(out)

def fit(model, Xtr, ytr, Xva, yva, epochs, lr, bs, weights=None, name=""):
    """Plain training loop. Keeps the epoch with the best validation macro-F1."""
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    w = None if weights is None else torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    lossf = nn.CrossEntropyLoss(weight=w)
    dl = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), batch_size=bs, shuffle=True)
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        f1 = f1_score(yva, predict(model, Xva), average="macro")
        print(f"  {name} epoch {ep + 1}/{epochs} | loss {tot / len(Xtr):.4f} | validation macro-F1 {f1:.3f}")
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model


Vocabulary size: 17395


In [32]:
class SectionCNN(nn.Module):
    """TextCNN: slides small windows (2, 3 and 4 words) over a line to spot patterns like 'Bachelor of ...'."""
    def __init__(self, vocab_size, n_cls, emb=100, filt=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(emb, filt, k, padding=k // 2) for k in (2, 3, 4)])
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(filt * 3, n_cls)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = torch.cat([torch.relu(c(e)).max(dim=2).values for c in self.convs], dim=1)
        return self.fc(self.drop(h))

lab_id = {l: i for i, l in enumerate(LABELS)}
lines = lines[lines["section"].isin(LABELS)].reset_index(drop=True)
X_all = encode(lines["line_anon"].tolist(), 40)
y_all = lines["section"].map(lab_id).values.astype(np.int64)
m = {s: (lines["split"] == s).values for s in ("train", "val", "test")}
freq = np.bincount(y_all[m["train"]], minlength=len(LABELS)).astype(float)
weights = (freq.sum() / np.maximum(freq, 1)) ** 0.5
weights = weights / weights.mean()                       # rare sections count more, so the model cannot just say "EXPERIENCE"
print("\nTraining the section CNN on", int(m["train"].sum()), "lines")
t0 = time.time()
cnn = fit(SectionCNN(len(vocab), len(LABELS)), X_all[m["train"]], y_all[m["train"]], X_all[m["val"]], y_all[m["val"]],
          epochs=5, lr=2e-3, bs=512, weights=weights, name="CNN")
cnn_pred = predict(cnn, X_all[m["test"]])
cnn_acc, cnn_f1 = accuracy_score(y_all[m["test"]], cnn_pred), f1_score(y_all[m["test"]], cnn_pred, average="macro")
cnn_rep = classification_report(y_all[m["test"]], cnn_pred, target_names=LABELS, output_dict=True, zero_division=0)
print(f"CNN done in {time.time() - t0:.0f}s")
torch.save(cnn.state_dict(), f"{MODELS}/section_cnn.pt")

# CNN output for every line -> share of each section per resume (a feature for the final DNN in Step 5)
all_pred = predict(cnn, X_all)
stats = pd.crosstab(lines["ID"], pd.Series(all_pred).map(dict(enumerate(LABELS))).values, normalize="index")
stats = stats.reindex(index=res["ID"], columns=LABELS).fillna(0.0)
np.save(f"{DATA}/section_stats.npy", stats.values.astype(np.float32))


Training the section CNN on 98921 lines
  CNN epoch 1/5 | loss 1.0157 | validation macro-F1 0.644
  CNN epoch 2/5 | loss 0.7676 | validation macro-F1 0.673
  CNN epoch 3/5 | loss 0.6718 | validation macro-F1 0.688
  CNN epoch 4/5 | loss 0.6013 | validation macro-F1 0.688
  CNN epoch 5/5 | loss 0.5474 | validation macro-F1 0.688
CNN done in 209s


In [ ]:
MAXL, MAXT = 30, 20
class CareerBiLSTM(nn.Module):
    """RNN (BiLSTM): turns each job line into a vector, then reads the lines in order, forwards and backwards."""
    def __init__(self, vocab_size, n_cls, emb=64, hid=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hid * 2, n_cls)
    def encode(self, x):                                     # x: [batch, lines, words]
        wmask = (x > 0).float().unsqueeze(-1)
        line_vec = (self.emb(x) * wmask).sum(2) / wmask.sum(2).clamp(min=1)
        out, _ = self.lstm(line_vec)
        lmask = (x.sum(2) > 0).float().unsqueeze(-1)
        return (out * lmask).sum(1) / lmask.sum(1).clamp(min=1)   # one "career vector" per resume
    def forward(self, x):
        return self.fc(self.drop(self.encode(x)))

Xc = np.zeros((len(res), MAXL, MAXT), dtype=np.int64)
used = collections.Counter()
exp = lines[lines["section"] == "EXPERIENCE"]
for rid, txt in zip(exp["ID"], exp["line_anon"]):
    i = pos.get(rid)
    if i is None or used[rid] >= MAXL:
        continue
    ids = [vocab.get(w, 1) for w in toks(txt)][:MAXT]
    Xc[i, used[rid], :len(ids)] = ids
    used[rid] += 1
cat_id = {c: i for i, c in enumerate(CATS)}
yc = res["Category"].map(cat_id).values.astype(np.int64)
sp = res["split"].values
print("\nTraining the career BiLSTM on", int((sp == "train").sum()), "resumes")
t0 = time.time()
lstm = fit(CareerBiLSTM(len(vocab), len(CATS)), Xc[sp == "train"], yc[sp == "train"], Xc[sp == "val"], yc[sp == "val"],
           epochs=15, lr=2e-3, bs=64, name="BiLSTM")
lstm_pred = predict(lstm, Xc[sp == "test"], bs=256)
lstm_acc, lstm_f1 = accuracy_score(yc[sp == "test"], lstm_pred), f1_score(yc[sp == "test"], lstm_pred, average="macro")
print(f"BiLSTM done in {time.time() - t0:.0f}s")
torch.save(lstm.state_dict(), f"{MODELS}/career_bilstm.pt")
lstm.eval(); vecs = []
with torch.no_grad():
    for i in range(0, len(Xc), 256):
        vecs.append(lstm.encode(torch.tensor(Xc[i:i + 256]).to(DEVICE)).cpu().numpy())
np.save(f"{DATA}/career_vecs.npy", np.concatenate(vecs).astype(np.float32))
json.dump({"labels": LABELS, "cats": CATS, "line_len": 40, "max_lines": MAXL, "max_words": MAXT}, open(f"{MODELS}/meta.json", "w"))


In [ ]:
def title(t):
    print("\n" + "=" * 8 + " " + t + " " + "=" * 8)

title("A. SECTION CNN (test lines)")
print(f"accuracy = {cnn_acc:.3f} | macro F1 = {cnn_f1:.3f}")
for l in LABELS:
    print(f"  {l:11s} F1 = {cnn_rep[l]['f1-score']:.2f}  (lines: {int(cnn_rep[l]['support'])})")

title("B. CAREER BiLSTM (test resumes, guesses the job category from the job history only)")
print(f"accuracy = {lstm_acc:.3f} | macro F1 = {lstm_f1:.3f} | chance level = {1 / len(CATS):.3f}")

title("C. SAVED FILES")
for d in (DATA, MODELS):
    print(d, {f: f"{os.path.getsize(os.path.join(d, f)) / 1e6:.1f} MB" for f in sorted(os.listdir(d))})

print("\nDONE")

In [35]:
import os, re, sys, json, time, random, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import joblib
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap

if "anonymize" not in globals() or "FIRST_NAMES" not in globals():
    raise RuntimeError("Run Step 4 first in this same notebook session (the anonymizer from Step 4 is needed here).")

DATA, MODELS = "/content/fairforge_data", "/content/fairforge_models"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)
METRICS = {}

def title(t):
    print("\n" + "=" * 8 + " " + t + " " + "=" * 8)

In [ ]:
res = pd.read_csv(f"{DATA}/resumes_anon.csv")
order = pd.read_csv(f"{DATA}/resume_order.csv")
res = res.set_index("ID").loc[order["ID"]].reset_index()                 # same row order as the saved arrays
res["text_raw"], res["text_anon"] = res["text_raw"].fillna("").astype(str), res["text_anon"].fillna("").astype(str)
emb_anon = np.load(f"{DATA}/emb_anon.npy")
sec = np.load(f"{DATA}/section_stats.npy")
car = np.load(f"{DATA}/career_vecs.npy")
CATS = sorted(res["Category"].unique())
y = res["Category"].map({c: i for i, c in enumerate(CATS)}).values.astype(np.int64)
sp = res["split"].values
TR, VA, TE = sp == "train", sp == "val", sp == "test"

# ================================================================ DNN (start)
class FitDNN(nn.Module):
    """DNN: three hidden layers that turn the resume features into a fit score for each job category."""
    def __init__(self, d_in, n_cls):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 512), nn.ReLU(), nn.Dropout(0.4),
                                 nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.4),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_cls))
    def forward(self, x):
        return self.net(x)

def run_dnn(X, epochs=80):
    model = FitDNN(X.shape[1], len(CATS)).to(DEVICE)
    cw = compute_class_weight("balanced", classes=np.arange(len(CATS)), y=y[TR])
    lossf = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    dl = DataLoader(TensorDataset(torch.tensor(X[TR]), torch.tensor(y[TR])), batch_size=64, shuffle=True)
    def pred(M):
        model.eval()
        with torch.no_grad():
            return model(torch.tensor(M).to(DEVICE)).argmax(1).cpu().numpy()
    best, best_state = -1.0, None
    for _ in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            lossf(model(xb), yb).backward()
            opt.step()
        f1 = f1_score(y[VA], pred(X[VA]), average="macro")
        if f1 > best:
            best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    p = pred(X[TE])
    return model, best, accuracy_score(y[TE], p), f1_score(y[TE], p, average="macro")

CONFIGS = {"MiniLM only": (False, False), "MiniLM + CNN sections": (True, False), "MiniLM + CNN + BiLSTM": (True, True)}
dnn_rows, best_cfg, best_val, best_model = [], None, -1.0, None
for name, (use_sec, use_car) in CONFIGS.items():
    parts = [emb_anon] + ([sec] if use_sec else []) + ([car] if use_car else [])
    X = np.hstack(parts).astype(np.float32)
    t0 = time.time()
    model, val_f1, acc, f1 = run_dnn(X)
    dnn_rows.append((name, val_f1, acc, f1, time.time() - t0))
    if val_f1 > best_val:
        best_cfg, best_val, best_model = (name, use_sec, use_car), val_f1, model
torch.save({k: v.cpu() for k, v in best_model.state_dict().items()}, f"{MODELS}/fit_dnn.pt")
json.dump({"features": best_cfg[0], "use_sections": best_cfg[1], "use_career": best_cfg[2], "categories": CATS},
          open(f"{MODELS}/fit_dnn_config.json", "w"))
# ================================================================ DNN (end)

# classic ML comparisons
lr_emb = LogisticRegression(max_iter=1000, C=10, class_weight="balanced").fit(emb_anon[TR], res["Category"][TR])
acc_lr_emb = accuracy_score(res["Category"][TE], lr_emb.predict(emb_anon[TE]))
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True, stop_words="english")
Xtf = tfidf.fit_transform(res["text_anon"][TR])
lr_tf = LogisticRegression(max_iter=300, C=10, class_weight="balanced").fit(Xtf, res["Category"][TR])
acc_tf = accuracy_score(res["Category"][TE], lr_tf.predict(tfidf.transform(res["text_anon"][TE])))
joblib.dump({"vectorizer": tfidf, "model": lr_tf}, f"{MODELS}/baseline_anon.joblib")
if os.path.exists(f"{MODELS}/baseline_raw.joblib"):
    os.remove(f"{MODELS}/baseline_raw.joblib")                              # not needed in the app, saves space
cent = np.stack([emb_anon[TR & (res["Category"].values == c)].mean(0) for c in CATS])
cent = cent / np.linalg.norm(cent, axis=1, keepdims=True)
np.save(f"{MODELS}/category_centroids.npy", cent.astype(np.float32))        # lets the app match a job description to job types

In [37]:
shap_text = []
try:
    vx = TfidfVectorizer(max_features=3000, stop_words="english", sublinear_tf=True, min_df=3)
    Xtr_x = vx.fit_transform(res["text_anon"][TR]).toarray().astype(np.float32)
    Xte_x = vx.transform(res["text_anon"][TE]).toarray().astype(np.float32)
    lr_x = LogisticRegression(max_iter=300, C=10, class_weight="balanced").fit(Xtr_x, res["Category"][TR])
    acc_x = accuracy_score(res["Category"][TE], lr_x.predict(Xte_x))
    bg = Xtr_x[np.random.RandomState(0).choice(len(Xtr_x), min(200, len(Xtr_x)), replace=False)]
    explainer = shap.LinearExplainer(lr_x, shap.maskers.Independent(bg, max_samples=len(bg)))
    te_cats = res["Category"][TE].values
    pick, seen_c = [], set()
    for i, c in enumerate(te_cats):
        if c not in seen_c and len(pick) < 3:
            pick.append(i); seen_c.add(c)
    sv = explainer.shap_values(Xte_x[pick])
    arr = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)
    if arr.ndim == 3 and arr.shape[0] == len(CATS) and arr.shape[2] != len(CATS):
        arr = np.transpose(arr, (1, 2, 0))
    words = vx.get_feature_names_out()
    for j, i in enumerate(pick):
        pred_label = lr_x.predict(Xte_x[i:i + 1])[0]
        c = list(lr_x.classes_).index(pred_label)
        vals = arr[j, :, c]
        up, down = np.argsort(-vals)[:6], np.argsort(vals)[:3]
        base = float(np.atleast_1d(explainer.expected_value)[c])
        gap = abs(base + vals.sum() - lr_x.decision_function(Xte_x[i:i + 1])[0, c])
        shap_text.append(f"[{te_cats[i]}] predicted {lr_x.classes_[c]} | words that raised it: {[words[k] for k in up]} | lowered it: {[words[k] for k in down]} | additivity gap {gap:.4f}")
    METRICS["shap_model_accuracy"] = round(float(acc_x), 3)
    np.save(f"{MODELS}/explain_bg.npy", bg)
    joblib.dump({"vectorizer": vx, "model": lr_x}, f"{MODELS}/explain_lr.joblib")
except Exception as e:
    shap_text.append("SHAP step skipped because of: " + repr(e)[:200] + "  (send me this line)")

In [ ]:
rng, nprng = random.Random(7), np.random.RandomState(7)
FEM_SEEN = ["Emily", "Sarah", "Priya", "Anjali", "Jessica", "Lakshmi", "Divya", "Pooja", "Neha", "Kavita", "Megan", "Rachel"]
MAL_SEEN = ["James", "Rahul", "Michael", "Amit", "David", "Arjun", "Karthik", "Vikram", "Robert", "Rohit", "Daniel", "Naveen"]
FEM_NEW = ["Kavya", "Meera", "Tanya", "Zoya", "Ishita", "Aarushi"]       # names NOT in the anonymizer's list: tests its real recall
MAL_NEW = ["Tanvir", "Dylan", "Harsh", "Yash", "Joel", "Reyansh"]
LASTS = ["Nair", "Sharma", "Iyer", "Reddy", "Carter", "Smith", "Johnson", "Khan", "Patel", "Singh", "Williams", "Brown", "Kumar", "Das"]

NAME_SHAPE = re.compile(r"[A-Z][a-z]+(?:\s+[A-Z]\.?)?(?:\s+[A-Z][a-z]+)*")

def anonymize_header(texts):
    """The candidate's name sits at the very top of a resume, so there ANY person-like name is hidden, not only known first names."""
    out = []
    for doc in nlp.pipe(texts, batch_size=500):
        t = doc.text
        spans = sorted(((e.start_char, e.end_char) for e in doc.ents
                        if e.label_ == "PERSON" and e.start_char < 60 and NAME_SHAPE.fullmatch(e.text.strip())
                        and not any(w.lower() in NOT_NAME for w in e.text.split())), reverse=True)
        for s_, e_ in spans:
            t = t[:s_] + "[NAME]" + t[e_:]
        out.append(t)
    return out

def pick_name(female):
    unseen = rng.random() < 0.3
    pool = (FEM_NEW if unseen else FEM_SEEN) if female else (MAL_NEW if unseen else MAL_SEEN)
    return rng.choice(pool), unseen

def year_for(old):
    return rng.randint(1998, 2005) if old else rng.randint(2018, 2023)

def snippet(first, last, female, year):
    return f"{first} {last}. {'She' if female else 'He'} graduated in {year}. "

# training history: every training resume gets a random synthetic identity and a hiring decision that is deliberately biased
KEYS = ["management", "manager", "led", "lead", "developed", "managed", "team", "project", "analysis", "responsible", "customer", "budget", "training", "design"]
merit = np.log1p(np.array([sum(t.lower().count(k) for k in KEYS) for t in res["text_raw"]]))
merit = (merit - merit[TR].mean()) / (merit[TR].std() + 1e-9)
tr_idx = np.where(TR)[0]
tr_id = []
for i in tr_idx:
    fem, old = rng.random() < 0.5, rng.random() < 0.5
    first, unseen = pick_name(fem)
    tr_id.append((first, rng.choice(LASTS), fem, year_for(old), old))
p_hire = 1 / (1 + np.exp(-(1.2 * merit[tr_idx] - 1.2 * np.array([m[2] for m in tr_id]) - 1.0 * np.array([m[4] for m in tr_id]))))
hired = (nprng.rand(len(tr_idx)) < p_hire).astype(int)                        # SYNTHETIC biased history (women and older graduates favoured less)

te_idx = np.where(TE)[0]
gen_pairs, age_pairs = [], []
for i in te_idx:
    last, yr = rng.choice(LASTS), year_for(rng.random() < 0.5)
    f, fu = pick_name(True); m, mu = pick_name(False)
    gen_pairs.append(((f, last, True, yr, fu), (m, last, False, yr, mu)))
    fem = rng.random() < 0.5; nm, nu = pick_name(fem)
    age_pairs.append(((nm, last, fem, year_for(True), nu), (nm, last, fem, year_for(False), nu)))

all_ids = [(m[0], m[1], m[2], m[3]) for m in tr_id] + [t[:4] for p in gen_pairs for t in p] + [t[:4] for p in age_pairs for t in p]
raw_snips = [snippet(*t) for t in all_ids]
anon_snips = anonymize(anonymize_header(raw_snips), ["EDUCATION"] * len(raw_snips))
nt, ng = len(tr_idx), 2 * len(te_idx)
tr_raw_s, tr_anon_s = raw_snips[:nt], anon_snips[:nt]
g_raw_s, g_anon_s = raw_snips[nt:nt + ng], anon_snips[nt:nt + ng]
a_raw_s, a_anon_s = raw_snips[nt + ng:], anon_snips[nt + ng:]

def texts(snips_, base_idx, version):
    col = res["text_raw"] if version == "raw" else res["text_anon"]
    return [s + col.iloc[b] for s, b in zip(snips_, base_idx)]

twin_base = np.repeat(te_idx, 2)                                             # each test resume appears as 2 twins
tr_raw_t, tr_anon_t = texts(tr_raw_s, tr_idx, "raw"), texts(tr_anon_s, tr_idx, "anon")
g_raw_t, g_anon_t = texts(g_raw_s, twin_base, "raw"), texts(g_anon_s, twin_base, "anon")
a_raw_t, a_anon_t = texts(a_raw_s, twin_base, "raw"), texts(a_anon_s, twin_base, "anon")

def make_scorer(train_texts):
    v = TfidfVectorizer(min_df=2, max_features=20000, sublinear_tf=True)
    clf = LogisticRegression(max_iter=300, C=5).fit(v.fit_transform(train_texts), hired)
    return lambda tx: clf.predict_proba(v.transform(tx))[:, 1]

score_raw, score_fair = make_scorer(tr_raw_t), make_scorer(tr_anon_t)       # same model, one sees identity, one does not

def audit(sa, sb, q=0.70):
    thr = np.quantile(np.concatenate([sa, sb]), q)
    ra, rb = float((sa >= thr).mean()), float((sb >= thr).mean())
    return {"rate_A": ra, "rate_B": rb, "DI": (min(ra, rb) / max(ra, rb)) if max(ra, rb) > 0 else 1.0,
            "DP_gap": abs(ra - rb), "flip": float(((sa >= thr) != (sb >= thr)).mean())}, thr

fair = {}
for kind, rt, at in (("gender", g_raw_t, g_anon_t), ("age", a_raw_t, a_anon_t)):
    for ver, sc, tx in (("raw", score_raw, rt), ("anonymized", score_fair, at)):
        s = sc(tx)
        fair[(kind, ver)], _ = audit(s[0::2], s[1::2])      # twin A (female / older) vs twin B (male / younger)

# adversary probe: can a model still guess gender or age?
def adversary(train_t, train_y, test_t, test_y):
    v = TfidfVectorizer(min_df=2, max_features=20000)
    clf = LogisticRegression(max_iter=300).fit(v.fit_transform(train_t), train_y)
    return accuracy_score(test_y, clf.predict(v.transform(test_t)))

yg_tr, ya_tr = [int(m[2]) for m in tr_id], [int(m[4]) for m in tr_id]
yg_te, ya_te = [1, 0] * len(te_idx), [1, 0] * len(te_idx)
leak = {("gender", "raw"): adversary(tr_raw_t, yg_tr, g_raw_t, yg_te), ("gender", "anonymized"): adversary(tr_anon_t, yg_tr, g_anon_t, yg_te),
        ("age", "raw"): adversary(tr_raw_t, ya_tr, a_raw_t, ya_te), ("age", "anonymized"): adversary(tr_anon_t, ya_tr, a_anon_t, ya_te)}

# anonymizer recall on the injected identities (test twins)
rec = {"name_seen": [], "name_new": [], "pronoun": [], "year": []}
for s_raw, s_anon, t in zip(g_raw_s, g_anon_s, [t for p in gen_pairs for t in p]):
    first, unseen, year = t[0], t[4], str(t[3])
    rec["name_new" if unseen else "name_seen"].append(first not in s_anon)
    rec["pronoun"].append(re.search(r"\b(She|He)\b", s_anon) is None)
    rec["year"].append(year not in s_anon)
identical = float(np.mean([g_anon_s[i] == g_anon_s[i + 1] for i in range(0, len(g_anon_s), 2)]))


In [ ]:
METRICS.update({"dnn_ablation": {r[0]: {"val_f1": round(r[1], 3), "test_acc": round(r[2], 3), "test_f1": round(r[3], 3)} for r in dnn_rows},
                "dnn_chosen": best_cfg[0], "tfidf_logreg_acc": round(acc_tf, 3), "minilm_logreg_acc": round(acc_lr_emb, 3),
                "fairness": {f"{k[0]}_{k[1]}": {a: round(b, 3) for a, b in v.items()} for k, v in fair.items()},
                "adversary_accuracy": {f"{k[0]}_{k[1]}": round(v, 3) for k, v in leak.items()},
                "anonymizer_recall": {k: round(float(np.mean(v)), 3) for k, v in rec.items() if v}, "twin_text_identical": round(identical, 3)})
json.dump(METRICS, open(f"{MODELS}/metrics.json", "w"), indent=2)
shutil.make_archive("/content/fairforge_models", "zip", MODELS)

In [40]:
title("A. DNN ('deep') vs CLASSIC ML: guess the job category (test set, 24 categories, chance = 4%)")
print(f"TF-IDF + Logistic Regression (ML baseline)  accuracy = {acc_tf:.3f}")
print(f"MiniLM + Logistic Regression (ML)           accuracy = {acc_lr_emb:.3f}")
for name, vf1, acc, f1, secs in dnn_rows:
    print(f"DNN on {name:24s} accuracy = {acc:.3f} | macro F1 = {f1:.3f} | validation F1 = {vf1:.3f} ({secs:.0f}s)")
print("Chosen for the app (best validation score):", best_cfg[0])

title("B. SHAP: WORDS THAT MOVED THE SCORE (3 test resumes)")
for line in shap_text:
    print(line)

title("C. ANONYMIZER RECALL ON INJECTED (SYNTHETIC) IDENTITIES")
for k, v in rec.items():
    if v:
        print(f"{k:10s} hidden in {100 * np.mean(v):5.1f}% of {len(v)} cases")
print(f"Twin pairs whose anonymized text is identical: {100 * identical:.1f}%")

title("D. FAIRNESS AUDIT: biased synthetic hiring history, selection = top 30% of twins")
print("(gender: A = female twin, B = male twin | age: A = older graduate, B = younger graduate)")
for k, v in fair.items():
    print(f"{k[0]:6s} {k[1]:11s} rate A = {v['rate_A']:.2f} | rate B = {v['rate_B']:.2f} | Disparate Impact = {v['DI']:.2f} "
          f"{'PASS' if v['DI'] >= 0.8 else 'FAIL'} | parity gap = {v['DP_gap']:.2f} | twin decision flips = {100 * v['flip']:.1f}%")

title("E. ADVERSARY: can a model guess gender / age? (50% = it cannot)")
for k, v in leak.items():
    print(f"{k[0]:6s} {k[1]:11s} accuracy = {v:.3f}")

title("F. SAVED FILES (zip made for GitHub)")
print(f"{MODELS}:", {f: f"{os.path.getsize(os.path.join(MODELS, f)) / 1e6:.1f} MB" for f in sorted(os.listdir(MODELS))})
print("Zip:", "/content/fairforge_models.zip", f"{os.path.getsize('/content/fairforge_models.zip') / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download("/content/fairforge_models.zip")
except Exception:
    print("Download the zip from the Files panel on the left (folder icon): fairforge_models.zip")

print("\nDONE")


======== A. DNN ('deep') vs CLASSIC ML: guess the job category (test set, 24 categories, chance = 4%) ========
TF-IDF + Logistic Regression (ML baseline)  accuracy = 0.654
MiniLM + Logistic Regression (ML)           accuracy = 0.601
DNN on MiniLM only              accuracy = 0.534 | macro F1 = 0.512 | validation F1 = 0.499 (12s)
DNN on MiniLM + CNN sections    accuracy = 0.547 | macro F1 = 0.518 | validation F1 = 0.493 (13s)
DNN on MiniLM + CNN + BiLSTM    accuracy = 0.475 | macro F1 = 0.435 | validation F1 = 0.492 (14s)
Chosen for the app (best validation score): MiniLM only

======== B. SHAP: WORDS THAT MOVED THE SCORE (3 test resumes) ========
[HR] predicted HR | words that raised it: ['hr', 'human', 'employee', 'compensation', 'hris', 'resources'] | lowered it: ['managing', 'spanish', 'teacher'] | additivity gap 0.0000
[DESIGNER] predicted DESIGNER | words that raised it: ['designer', 'designing', 'web', 'design', 'jewelry', 'art'] | lowered it: ['economics', 'organizing', 'food']

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE
